# 07 Coherence-resynthesis term: pass and fail cases

This notebook isolates `PhysicsComponents.coherence_resynthesis`. The term re-synthesises the per-baseline complex coherence from each profile (steering projection normalised by mass) and penalises their squared difference. It is the link between the height-domain fit and the interferometric observables.

Modules exercised: `PhysicsComponents.coherence_resynthesis`, `TomoGeometry`.

We confirm the term is near zero for a self-comparison and grows when the profile's mean or spread changes, and we visualise the resynthesised coherence per track for matched and mismatched profiles, alongside the per-track diagnostic from `PhysicsQuantitiesCheck._coherence_agreement`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from configuration.training_config import GeometryConfig
from tools.sar.tomo_geometry            import TomoGeometry
from tools.data.gaussians                import GaussianMixture
from pipelines.backbone_pipeline.loss import PhysicsComponents

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])
xt     = torch.tensor(x_axis, device=DEVICE)
geom   = TomoGeometry(GeometryConfig(), xt)
floor  = 1e-3

def resyn_coherence(profile):
    t  = to_check_tensor(profile[None, :])
    p0 = t.sum(dim=1) * dx
    g  = torch.einsum("nk,bkhw->bnhw", geom.steering, t.to(geom.steering.dtype)) * dx
    g  = g / p0.clamp(min=floor).unsqueeze(1)
    return g[0, :, 0, 0]

def coh_loss(pred, target):
    return float(PhysicsComponents.coherence_resynthesis(to_check_tensor(pred[None, :]), to_check_tensor(target[None, :]), geom.steering, dx, floor))


## Resynthesised coherence magnitude per track

We compare a target profile against a matched copy and against a mean-shifted profile. The coherence magnitude decays with baseline (longer baselines see more vertical structure); a mean shift mainly rotates phase, while a spread change attenuates magnitude.

In [ ]:
target  = gaussian_profile(x_axis, 1.0, 8.0, 4.0)
shifted = gaussian_profile(x_axis, 1.0, 16.0, 4.0)
widened = gaussian_profile(x_axis, 1.0, 8.0, 8.0)

ct = resyn_coherence(target)
cs = resyn_coherence(shifted)
cw = resyn_coherence(widened)
kz = geom.kz.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.4))
axes[0].plot(kz, ct.abs().cpu().numpy(), "o-", color="k",  label="target")
axes[0].plot(kz, cs.abs().cpu().numpy(), "s-", color="C1", label="mean-shift")
axes[0].plot(kz, cw.abs().cpu().numpy(), "^-", color="C0", label="widened")
axes[0].set_xlabel("kz [rad/m]")
axes[0].set_ylabel("|coherence|")
axes[0].set_title("resynthesised coherence magnitude")
axes[0].legend()
axes[1].plot(kz, np.angle(ct.cpu().numpy()), "o-", color="k",  label="target")
axes[1].plot(kz, np.angle(cs.cpu().numpy()), "s-", color="C1", label="mean-shift")
axes[1].set_xlabel("kz [rad/m]")
axes[1].set_ylabel("coherence phase [rad]")
axes[1].set_title("resynthesised coherence phase")
axes[1].legend()
fig.tight_layout()
plt.show()

## Term value: pass versus fail

The squared-difference loss should sit at the floor for a self-comparison and rise for both the mean-shifted and widened profiles.

In [ ]:
print(f"coherence_resyn(target,  target) = {coh_loss(target, target):.6e}")
print(f"coherence_resyn(shifted, target) = {coh_loss(shifted, target):.6e}")
print(f"coherence_resyn(widened, target) = {coh_loss(widened, target):.6e}")

shifts = np.linspace(0.0, 20.0, 21)
vals   = [coh_loss(gaussian_profile(x_axis, 1.0, 8.0 + s, 4.0), target) for s in shifts]

fig, ax = plt.subplots(figsize=(6.5, 3.3))
ax.plot(shifts, vals, "o-", color="C3")
ax.set_xlabel("mean shift of predicted profile [m]")
ax.set_ylabel("coherence_resyn term")
ax.set_title("Coherence-resynthesis penalty vs mean shift")
plt.show()

## Expected outcome

The matched profile gives a coherence-resynthesis term at the floor; both the mean-shifted and widened profiles raise it. The magnitude curves decay with `kz`, the mean shift rotates the coherence phase, and the penalty grows monotonically with the mean shift, confirming the term ties the height fit to the interferometric coherence.